In [1]:
# ============================================================
# Cell 1: 环境初始化
# ============================================================
import os, sys, json

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.ci.competitor_brief import generate_competitor_brief
from src.engineering_intelligence.technology_brief import generate_technology_brief
from src.engineering_intelligence.strategy_classifier import get_all_strategies

driver = get_driver()
print("✅ 环境就绪")

✅ 环境就绪


In [2]:
# ============================================================
# Cell 2: 生成带 landscape 的 Competitor Brief
# ============================================================
with driver.session() as s:
    org = s.run("""
        MATCH (o:Organization)-[:DEVELOPS]->(:DevelopmentProgram)
        RETURN o.organization_id AS oid, o.canonical_name AS name
        ORDER BY o.canonical_name LIMIT 1
    """).single()

org_id = org["oid"]
org_name = org["name"]
print(f"使用组织: {org_name} ({org_id})")

brief = generate_competitor_brief(driver, org_id, days_back=730, persist=True)
bid = brief["brief_id"]
print(f"✅ Competitor Brief: {bid}")

使用组织: Adaptive Phage Therapeutics (CI:ORG:55DF2E8C)
✅ Competitor Brief: CI:BRIEF:24BAC738


In [3]:
# ============================================================
# Cell 3: 打印 Competitor Brief 新增区块（white_space + landscape_matrix）
# ============================================================
# 从数据库读回，验证字段确实落库
with driver.session() as s:
    row = s.run("""
        MATCH (b:IntelligenceProduct {brief_id: $bid})
        RETURN b.white_space_analysis AS ws, b.landscape_matrix AS lm
    """, bid=bid).single()

ws_raw = row.get("ws") or "{}"
lm_raw = row.get("lm") or "{}"
ws = json.loads(ws_raw) if isinstance(ws_raw, str) else ws_raw
lm = json.loads(lm_raw) if isinstance(lm_raw, str) else lm_raw

print("=== white_space_analysis ===")
print("summary:", json.dumps(ws.get("summary", {}), ensure_ascii=False, indent=2))
print("pathogen_scope:", ws.get("pathogen_scope"))
print("top combinations:")
for c in ws.get("underrepresented_combinations", [])[:5]:
    print(f"  {c['strategy_type']:30s} × {c['pathogen_species']:30s} [{c['coverage']}]")
print("disclaimer:", ws.get("disclaimer", "")[:80])

print("\n=== landscape_matrix ===")
print("x_axis:", lm.get("x_axis"))
print("y_axis:", lm.get("y_axis"))
for row_cells in lm.get("cells", []):
    for cell in row_cells:
        if cell.get("count", 0) > 0:
            print(f"  {cell['strategy_type']} × {cell['pathogen_species']} -> {cell['count']} 个项目")

print("\n=== recommended_next_steps ===")
for step in brief.get("recommended_next_steps", []):
    print(f"  - {step}")

=== white_space_analysis ===
summary: {
  "total_combinations": 15,
  "underrepresented_count": 15,
  "empty_count": 6,
  "minimal_count": 9,
  "sparse_count": 0,
  "covered_count": 0
}
pathogen_scope: ['Pseudomonas aeruginosa', 'Staphylococcus aureus', 'Klebsiella pneumoniae']
top combinations:
  host_range_expansion           × Staphylococcus aureus          [empty]
  tail_fiber_engineering         × Pseudomonas aeruginosa         [empty]
  tail_fiber_engineering         × Staphylococcus aureus          [empty]
  lysis_enhancement              × Klebsiella pneumoniae          [empty]
  biofilm_disruption             × Klebsiella pneumoniae          [empty]
disclaimer: White space 表示在当前已收集、截至指定日期的公开证据中覆盖较少的区域。它不等于：市场上绝对无人布局、不存在未公开项目、该区域必然有商业价值、该区域不

=== landscape_matrix ===
x_axis: []
y_axis: ['Klebsiella pneumoniae', 'Pseudomonas aeruginosa', 'Staphylococcus aureus']

=== recommended_next_steps ===
  - 建议跟进近期重大事件，评估对内部战略的影响
  - 对手拥有 3 个专利族，建议启动 FTO 初筛
  - 对手有 1 项活跃临床试验，建议持续跟踪
  - 识别到

In [4]:
# ============================================================
# Cell 4: 生成带策略对比 + white-space 的 Technology Brief
# ============================================================
strategies = get_all_strategies(driver)
if not strategies:
    raise RuntimeError("尚无工程策略，请先跑 06 notebook")

target = strategies[0]
print(f"使用策略: {target['strategy_type']} ({target['id']})")

t_brief = generate_technology_brief(
    driver,
    strategy_id=target["id"],
    technology_question=f"{target['strategy_type']} 的公开证据成熟度如何？",
    persist=True,
)
t_bid = t_brief["brief_id"]
print(f"✅ Technology Brief: {t_bid}")

使用策略: biofilm_disruption (ENG:STRAT:B7FD45B5)
✅ Technology Brief: CI:BRIEF:B374EF8B


In [5]:
# ============================================================
# Cell 5: 打印 Technology Brief 新增区块
# ============================================================
with driver.session() as s:
    row = s.run("""
        MATCH (b:IntelligenceProduct {brief_id: $bid})
        RETURN b.strategy_comparison AS sc, b.white_space_analysis AS ws
    """, bid=t_bid).single()

sc_raw = row.get("sc") or "{}"
ws_raw = row.get("ws") or "{}"
sc = json.loads(sc_raw) if isinstance(sc_raw, str) else sc_raw
ws = json.loads(ws_raw) if isinstance(ws_raw, str) else ws_raw

print("=== strategy_comparison ===")
print("focus_strategy_ids:", sc.get("focus_strategy_ids"))
print("comparison_matrix:")
print(json.dumps(sc.get("comparison_matrix", {}), ensure_ascii=False, indent=2)[:1200])
print("\nstrategies 摘要:")
for s_ in sc.get("strategies", []):
    print(f"  {s_['strategy_type']:30s} constructs={s_['construct_count']} "
          f"maturity={s_['evidence_maturity']} orgs={len(s_['known_organizations'])}")

print("\n=== white_space_analysis ===")
print("summary:", json.dumps(ws.get("summary", {}), ensure_ascii=False, indent=2))
print("pathogen_scope:", ws.get("pathogen_scope"))
print("technology_scope:", ws.get("technology_scope"))

print("\n=== recommended_next_steps ===")
for step in t_brief.get("recommended_next_steps", []):
    print(f"  - {step}")

=== strategy_comparison ===
focus_strategy_ids: ['ENG:STRAT:B7FD45B5']
comparison_matrix:
{
  "strategy_types": [
    "biofilm_disruption",
    "host_range_expansion",
    "lysis_enhancement",
    "payload_delivery",
    "tail_fiber_engineering"
  ],
  "construct_count": [
    2,
    2,
    2,
    2,
    2
  ],
  "evidence_maturity": [
    "in_vitro",
    "in_vitro",
    "in_vitro",
    "in_vitro",
    "in_vitro"
  ],
  "organization_count": [
    1,
    1,
    1,
    1,
    1
  ],
  "patent_count": [
    3,
    3,
    3,
    3,
    3
  ]
}

strategies 摘要:
  biofilm_disruption             constructs=2 maturity=in_vitro orgs=1
  host_range_expansion           constructs=2 maturity=in_vitro orgs=1
  lysis_enhancement              constructs=2 maturity=in_vitro orgs=1
  payload_delivery               constructs=2 maturity=in_vitro orgs=1
  tail_fiber_engineering         constructs=2 maturity=in_vitro orgs=1

=== white_space_analysis ===
summary: {
  "total_combinations": 4,
  "underrepres

In [6]:
# ============================================================
# Cell 6: 全量验证 —— 所有 Competitor / Technology Brief 都有新字段
# ============================================================
with driver.session() as s:
    for btype in ["competitor", "engineered_phage_technology"]:
        print(f"\n=== {btype} ===")
        rows = list(s.run("""
            MATCH (b:IntelligenceProduct {brief_type: $bt})
            RETURN b.brief_id AS bid,
                   b.white_space_analysis AS ws,
                   b.landscape_matrix AS lm,
                   b.strategy_comparison AS sc
        """, bt=btype))
        for r in rows:
            has_ws = bool(r.get("ws") and r["ws"] not in ("", "{}"))
            has_lm = bool(r.get("lm") and r["lm"] not in ("", "{}"))
            has_sc = bool(r.get("sc") and r["sc"] not in ("", "{}"))
            print(f"  {r['bid']}  ws={has_ws}  lm={has_lm}  sc={has_sc}")


=== competitor ===
  CI:BRIEF:F8B1ED66  ws=False  lm=False  sc=False
  CI:BRIEF:42F0AEC9  ws=False  lm=False  sc=False
  CI:BRIEF:67584C1D  ws=False  lm=False  sc=False
  CI:BRIEF:B58E9AF7  ws=False  lm=False  sc=False
  CI:BRIEF:1A249DEF  ws=False  lm=False  sc=False
  CI:BRIEF:8A191620  ws=False  lm=False  sc=False
  CI:BRIEF:24BAC738  ws=True  lm=True  sc=False

=== engineered_phage_technology ===
  CI:BRIEF:A2E4D1DB  ws=False  lm=False  sc=False
  CI:BRIEF:11ABC6EF  ws=False  lm=False  sc=False
  CI:BRIEF:325517E6  ws=False  lm=False  sc=False
  CI:BRIEF:54E0A2D4  ws=False  lm=False  sc=False
  CI:BRIEF:7C8AD329  ws=False  lm=False  sc=False
  CI:BRIEF:8DCF54F5  ws=False  lm=False  sc=False
  CI:BRIEF:7AF50CD1  ws=False  lm=False  sc=False
  CI:BRIEF:895793D3  ws=False  lm=False  sc=False
  CI:BRIEF:B374EF8B  ws=True  lm=False  sc=True
